# HF SPECTER MWE

Dieses Notebook ist ein **MWE** (*Minimal Working Example*): das kleinstmoegliche, aber lauffaehige Beispiel fuer `InferenceClient.feature_extraction(...)` mit `allenai/specter`.

Es zeigt:
- Laden von `HF_TOKEN` aus `/home/ubuntu/Author_Name_Disambiguation/.env`
- Aufbau eines laengeren wissenschaftlichen Beispieltexts aus **Title + Abstract**
- Aufruf von `huggingface_hub.InferenceClient`
- Kurze Inspektion der Rueckgabeform

Das Repo nutzt produktiv denselben SPECTER-Vertrag (`Title [SEP] Abstract`), hat dort aber zusaetzlich Retry- und Shape-Normalisierung.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
from huggingface_hub import InferenceClient

REPO_ROOT = Path('/home/ubuntu/Author_Name_Disambiguation')
DOTENV_PATH = REPO_ROOT / '.env'


def load_env_value(env_path: Path, key: str) -> str:
    if not env_path.exists():
        raise FileNotFoundError(f'Missing env file: {env_path}')

    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue

        name, value = line.split('=', 1)
        if name.strip() != key:
            continue

        parsed = value.strip().strip("\"").strip("'")
        if not parsed:
            break
        os.environ[key] = parsed
        return parsed

    raise KeyError(f'{key} not found in {env_path}')


HF_TOKEN = os.environ.get('HF_TOKEN') or load_env_value(DOTENV_PATH, 'HF_TOKEN')
print(f'HF_TOKEN loaded from {DOTENV_PATH}: yes, length={len(HF_TOKEN)}')

HF_TOKEN loaded from /home/ubuntu/Author_Name_Disambiguation/.env: yes, length=37


In [2]:
title = 'Graph-Aware Author Name Disambiguation with Citation Context and Transformer-Based Document Embeddings'
abstract = (
    'Author name disambiguation remains a critical bottleneck for the construction of reliable scholarly knowledge graphs, '
    'especially in large-scale bibliographic collections where homonyms, inconsistent initials, and affiliation drift are common. '
    'We present a graph-aware disambiguation pipeline that combines transformer-based document embeddings with citation-context signals, '
    'coauthor overlap, venue trajectories, and temporally constrained affiliation evidence. '
    'Our approach represents each publication as a fused document vector derived from title and abstract text, then augments pairwise scoring '
    'with neighborhood features extracted from local citation and collaboration graphs. '
    'In experiments on astronomy and computer-science benchmarks, the proposed method improves cluster purity and pairwise F1 over strong '
    'blocking-based baselines while remaining robust under sparse metadata conditions. '
    'A detailed ablation study shows that document-level semantic representations are particularly valuable when lexical name evidence is weak, '
    'and that graph cues reduce false merges among prolific authors publishing across adjacent subfields.'
)


def build_source_text(title: str, abstract: str) -> str:
    title = (title or '').strip()
    abstract = (abstract or '').strip()
    if title and abstract:
        return f'{title} [SEP] {abstract}'
    return title or abstract


paper_text = build_source_text(title, abstract)
print(title)
print()
print(f'Abstract characters: {len(abstract)}')
print(f'Combined request characters: {len(paper_text)}')
print()
print(paper_text[:500] + '...')

Graph-Aware Author Name Disambiguation with Citation Context and Transformer-Based Document Embeddings

Abstract characters: 1136
Combined request characters: 1245

Graph-Aware Author Name Disambiguation with Citation Context and Transformer-Based Document Embeddings [SEP] Author name disambiguation remains a critical bottleneck for the construction of reliable scholarly knowledge graphs, especially in large-scale bibliographic collections where homonyms, inconsistent initials, and affiliation drift are common. We present a graph-aware disambiguation pipeline that combines transformer-based document embeddings with citation-context signals, coauthor overlap...


In [3]:
client = InferenceClient(
    provider='hf-inference',
    api_key=os.environ['HF_TOKEN'],
)

result = client.feature_extraction(
    paper_text,
    model='allenai/specter',
)

print(f'Response Python type: {type(result).__name__}')

Response Python type: ndarray


In [5]:
response_array = np.asarray(result, dtype=np.float32)

if response_array.ndim == 1:
    vector = response_array
    vector_strategy = 'already 1D'
elif response_array.ndim == 2:
    vector = response_array[0]
    vector_strategy = 'first token / CLS row'
elif response_array.ndim == 3:
    vector = response_array[0, 0]
    vector_strategy = 'batch[0], first token / CLS row'
else:
    raise ValueError(f'Unexpected response shape: {response_array.shape}')

summary = {
    'raw_shape': list(response_array.shape),
    'vector_shape': list(vector.shape),
    'vector_dtype': str(vector.dtype),
    'vector_strategy': vector_strategy,
    'vector_preview': vector[:10].round(6).tolist(),
    'vector_norm_l2': float(np.linalg.norm(vector)),
}

print('Note: hf-inference returns token-level features for this request; the repo uses the first token as the document embedding.')
print(json.dumps(summary, indent=2))
vector[:10]

Note: hf-inference returns token-level features for this request; the repo uses the first token as the document embedding.
{
  "raw_shape": [
    213,
    768
  ],
  "vector_shape": [
    768
  ],
  "vector_dtype": "float32",
  "vector_strategy": "first token / CLS row",
  "vector_preview": [
    -0.9996240139007568,
    1.1690900325775146,
    0.7262930274009705,
    -0.27566400170326233,
    0.7254520058631897,
    -0.7467550039291382,
    -0.13463500142097473,
    0.10361699759960175,
    0.26190999150276184,
    0.24876999855041504
  ],
  "vector_norm_l2": 22.541824340820312
}


array([-0.99962366,  1.1690905 ,  0.726293  , -0.2756641 ,  0.72545165,
       -0.7467554 , -0.13463487,  0.10361686,  0.26191002,  0.24877027],
      dtype=float32)